In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../..")

In [2]:
import os
import json
import logging

import numpy as np
import torch
from rich.console import Console
from rich.theme import Theme

from src.models import ModelandTokenizer
from src.utils import env_utils, logging_utils

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4,5,6,7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging_utils.configure(level=logging.INFO)
console = Console(theme=Theme({"default": "white"}), highlight=False)


def rich_print(*a, **k):
    console.print(*a, **k)


MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
MODEL_SHORT = MODEL_NAME.split("/")[-1]
VERSION = "conditions"
CACHE_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "multistep_clock", VERSION, MODEL_SHORT
)

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(
    f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}"
)
print(
    f"residual={mt.layer_name_format}  mlp={mt.mlp_module_name_format}  attn={mt.attn_module_name_format}"
)

2026-06-15 11:07:28 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-06-15 11:07:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 11:07:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-06-15 11:07:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 11:07:29 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-15 11:07:29 httpx INFO     HTT

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-06-15 11:07:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 11:07:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-06-15 11:07:53 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40
residual=model.layers.{}  mlp=model.layers.{}.mlp  attn=model.layers.{}.self_attn


In [72]:
from src.functional import generate_with_patch
from src.tokens import prepare_input

system_prompt = "You are a helpful assistant"
query_template = """Checkout the following objects. For each object you should give its superclass, in the order they are given. You should use bullet points in your answer and only give the answer without any explanation.
Objects: {}"""

source_objects = ["Bus", "Dog", "Cricket", "Shoes", "Chair"]
source_prefix = """* Vehicle
* Animal
*"""
source_query = query_template.format(", ".join(source_objects))

destination_objects = ["Banana", "Watch", "Sofa", "Ball", "Rose"]
destination_prefix = """* Fruit
*"""
destination_query = query_template.format(", ".join(destination_objects))

# source_query, destination_query = destination_query, source_query
# source_prefix, destination_prefix = destination_prefix, source_prefix


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": source_query},
    {"role": "assistant", "content": source_prefix},
]
source_prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,  # Don't add a new assistant header
    continue_final_message=True,  # Don't close the assistant turn with <|eot_id|>
    tokenize=False,
    chat_template_kwargs={"enable_thinking": False},  # only Qwen3 and Qwen3.5
)
source_gen = generate_with_patch(
    mt,
    inputs=prepare_input(source_prompt, mt).to(mt.model.device),
    n_gen_per_prompt=1,
    max_new_tokens=16,
    do_sample=False,
    remove_prefix=True
)[0]
rich_print(f"[white]{source_prompt}[/white][salmon1]{source_gen}[/salmon1]")


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": destination_query},
    {"role": "assistant", "content": destination_prefix},
]
destination_prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,  # Don't add a new assistant header
    continue_final_message=True,  # Don't close the assistant turn with <|eot_id|>
    tokenize=False,
    chat_template_kwargs={"enable_thinking": False},  # only Qwen3 and Qwen3.5
)
destination_gen = generate_with_patch(
    mt,
    inputs=prepare_input(destination_prompt, mt).to(mt.model.device),
    n_gen_per_prompt=1,
    max_new_tokens=16,
    do_sample=False,
    remove_prefix=True
)[0]
rich_print(f"[white]{destination_prompt}[/white][salmon1]{destination_gen}[/salmon1]")

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Checkout the following objects. For each object you should give its superclass, in the order they are given. You 
should use bullet points in your answer and only give the answer without any explanation.
Objects: Bus, Dog, Cricket, Shoes, Chair<|im_end|>
<|im_start|>assistant
* Vehicle
* Animal
* Sport
* Footwear
* Furniture

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Checkout the following objects. For each object you should give its superclass, in the order they are given. You 
should use bullet points in your answer and only give the answer without any explanation.
Objects: Banana, Watch, Sofa, Ball, Rose<|im_end|>
<|im_start|>assistant
* Fruit
* Timepiece
* Furniture
* SportsEquipment
* Flower

In [73]:
from src.functional import get_hs, interpret_logits

# source run
source_inputs = prepare_input(source_prompt, mt)
logit_location = (mt.lm_head_name, -1)

source_hs = get_hs(
    mt = mt,
    input = source_inputs,
    locations = [(mt.layer_name_format.format(layer_idx), -1) for layer_idx in range(mt.n_layer)] + [logit_location]
)
source_logits = source_hs[logit_location].squeeze()
src_out = interpret_logits(mt, logits=source_logits)
print(f"source output: {[str(pred) for pred in src_out]}")

# destination run
destination_inputs = prepare_input(destination_prompt, mt)
destination_logits = get_hs(
    mt = mt,
    input = destination_inputs,
    locations = (mt.lm_head_name, -1)
).squeeze()
dst_out, base_track = interpret_logits(mt, logits=destination_logits, interested_tokens=(src_out[0].token_id,))
print(f"destination output: {[str(pred) for pred in dst_out]}")
base_track

source output: ["' Sport' [18207] (p=0.758, logit=25.125)", "' Sports' [13180] (p=0.218, logit=23.875)", "' Game' [4050] (p=0.010, logit=20.750)", "' In' [758] (p=0.010, logit=20.750)", "' Cricket' [57919] (p=0.001, logit=18.500)"]
destination output: ["' Time' [4120] (p=0.621, logit=22.375)", "' Access' [9549] (p=0.157, logit=21.000)", "' Electronic' [34169] (p=0.058, logit=20.000)", "' Electronics' [37684] (p=0.058, logit=20.000)", "' Device' [13903] (p=0.027, logit=19.250)"]


OrderedDict([(18207,
              (1151,
               PredictedToken(token=' Sport', prob=1.150183379650116e-07, logit=6.875, token_id=18207, metadata=None)))])

In [74]:
from src.utils.typing import PathLike, PredictedToken, TokenizerOutput
from src.functional import PatchSpec
from src.trace import get_window, patched_run


def patched_run(
    mt: ModelandTokenizer,
    clean_input: TokenizerOutput,
    patch_hs: dict,
    layer_idx: int,
    layer_name_format: str = mt.layer_name_format,
    window_size: int = 5,
):
    window = get_window(layer_name_format, layer_idx, window_size, mt.n_layer)
    patches = [
        PatchSpec(
            location=(layer_name, -1),
            patch=patch_hs[(layer_name, -1)],
            strategy="replace",
            on_pass="first",
        )
        for layer_name in window
    ]
    return get_hs(
        mt=mt,
        input=clean_input,
        locations=logit_location,
        patches=patches,
        return_dict=False,
    ).squeeze()


# smoke test
patched_logits = patched_run(
    mt=mt,
    clean_input=destination_inputs,
    patch_hs=source_hs,
    layer_idx=10,
    window_size=10,
)
patched_out, tracked_tokens = interpret_logits(
    mt,
    logits=patched_logits,
    interested_tokens=(src_out[0].token_id, dst_out[0].token_id),
)
print(f"patched output: {[str(pred) for pred in patched_out]}")
tracked_tokens

patched output: ["' Time' [4120] (p=0.641, logit=22.750)", "' Access' [9549] (p=0.184, logit=21.500)", "' Electronics' [37684] (p=0.052, logit=20.250)", "' Electronic' [34169] (p=0.046, logit=20.125)", "' Wear' [41062] (p=0.019, logit=19.250)"]


OrderedDict([(4120,
              (1,
               PredictedToken(token=' Time', prob=0.640625, logit=22.75, token_id=4120, metadata=None))),
             (18207,
              (1062,
               PredictedToken(token=' Sport', prob=9.546056389808655e-08, logit=7.03125, token_id=18207, metadata=None)))])

In [75]:
# sweep over layers

WINDOW_SIZE = 11

for layer_idx in range(0, mt.n_layer, 2):
    patched_logits = patched_run(
        mt=mt,
        clean_input=destination_inputs,
        patch_hs=source_hs,
        layer_idx=layer_idx,
        window_size=WINDOW_SIZE
    )
    patched_out = interpret_logits(mt, logits=patched_logits)
    rich_print(f"[cyan]Layer {layer_idx}[/cyan]: {[str(pred) for pred in patched_out]}")

Layer 0: ["' Time' [4120] (p=0.625, logit=22.500)", "' Access' [9549] (p=0.158, logit=21.125)", "' Electronic' 
[34169] (p=0.058, logit=20.125)", "' Electronics' [37684] (p=0.058, logit=20.125)", "' Device' [13903] (p=0.027, 
logit=19.375)"]

Layer 2: ["' Time' [4120] (p=0.641, logit=22.625)", "' Access' [9549] (p=0.162, logit=21.250)", "' Electronic' 
[34169] (p=0.053, logit=20.125)", "' Electronics' [37684] (p=0.053, logit=20.125)", "' Device' [13903] (p=0.022, 
logit=19.250)"]

Layer 4: ["' Time' [4120] (p=0.621, logit=22.500)", "' Access' [9549] (p=0.178, logit=21.250)", "' Electronics' 
[37684] (p=0.058, logit=20.125)", "' Electronic' [34169] (p=0.051, logit=20.000)", "' Device' [13903] (p=0.021, 
logit=19.125)"]

Layer 6: ["' Time' [4120] (p=0.648, logit=22.625)", "' Access' [9549] (p=0.163, logit=21.250)", "' Electronics' 
[37684] (p=0.053, logit=20.125)", "' Electronic' [34169] (p=0.047, logit=20.000)", "' Device' [13903] (p=0.022, 
logit=19.250)"]

Layer 8: ["' Time' [4120] (p=0.648, logit=22.625)", "' Access' [9549] (p=0.164, logit=21.250)", "' Electronics' 
[37684] (p=0.053, logit=20.125)", "' Electronic' [34169] (p=0.047, logit=20.000)", "' Wear' [41062] (p=0.022, 
logit=19.250)"]

Layer 10: ["' Time' [4120] (p=0.641, logit=22.750)", "' Access' [9549] (p=0.184, logit=21.500)", "' Electronics' 
[37684] (p=0.052, logit=20.250)", "' Electronic' [34169] (p=0.046, logit=20.125)", "' Wear' [41062] (p=0.019, 
logit=19.250)"]

... (22 outputs left out)

Layer 56: ["' Sport' [18207] (p=0.754, logit=24.625)", "' Sports' [13180] (p=0.216, logit=23.375)", "' Game' [4050]
(p=0.011, logit=20.375)", "' In' [758] (p=0.009, logit=20.250)", "' Cricket' [57919] (p=0.005, logit=19.625)"]

Layer 58: ["' Sport' [18207] (p=0.758, logit=25.125)", "' Sports' [13180] (p=0.218, logit=23.875)", "' Game' [4050]
(p=0.010, logit=20.750)", "' In' [758] (p=0.010, logit=20.750)", "' Cricket' [57919] (p=0.001, logit=18.500)"]

Layer 60: ["' Sport' [18207] (p=0.758, logit=25.125)", "' Sports' [13180] (p=0.218, logit=23.875)", "' Game' [4050]
(p=0.010, logit=20.750)", "' In' [758] (p=0.010, logit=20.750)", "' Cricket' [57919] (p=0.001, logit=18.500)"]

Layer 62: ["' Sport' [18207] (p=0.758, logit=25.125)", "' Sports' [13180] (p=0.218, logit=23.875)", "' Game' [4050]
(p=0.010, logit=20.750)", "' In' [758] (p=0.010, logit=20.750)", "' Cricket' [57919] (p=0.001, logit=18.500)"]